Goal (plain language): turn your team's coding rules and agent boundaries into two files that AI assistants can actually follow — and prove they exist and are well-formed with a small validator you can run locally.

Roadmap — what the notebook does, then what you'll understand:
- What we'll do: first we check your computer can run the steps; then we set a target repo (or make a safe demo one); then we author two Markdown files that guide AI assistants (repo rules + agent personas); then we write a tiny Python validator and run it; finally we commit the files locally. No network calls or paid services.
- What you'll understand: how repo-level guidance (Copilot instructions) differs from per-prompt tips, how to express Python repo conventions clearly, how to define agent roles with allowed/forbidden actions, and how to enforce the presence/structure of these docs via a simple script (and how to wire it into CI later).

Assumptions and your likely gap:
- Assumes: basic Python, Markdown, and git add/commit familiarity.
- Likely gaps we will teach: the standard Python repo layout (src/, tests/), what a repository-level Copilot instructions file is meant to control, how to define agent personas with boundaries, and how to do minimal programmatic Markdown validation.

Note on artifacts: this lesson builds real files in a git repo and validates them. By default, it creates a demo repo next to this notebook so you can run safely; you can point it at your own repo if you prefer.

Pipeline map — what we will build and verify

```text
[Setup check]
     |
[Select or create target repo]
     |
[Write .github/copilot-instructions.md]
     |--> [Validate structure: headings, examples, tokens, forbidden patterns]
     |
[Write AGENTS.md]
     |--> [Validate structure: agent block + sections, templates, interactions]
     |
[Write tools/validate_agent_docs.py]
     |--> [Run validator script] --> [JSON + summary output]
     |
[Commit artifacts locally]
```
Use this map as a reference: each dense cell below calls out which box it implements. Any files written are explained right after they are created (what, where, and why).

In [1]:
# Setup check — run me first
import importlib, sys, subprocess, shutil
from pathlib import Path

REQUIRED = {
    # stdlib only in this lesson
}
missing = [f"{m} ({hint})" for m, hint in REQUIRED.items() if importlib.util.find_spec(m) is None]
if missing:
    raise SystemExit("Missing prerequisites:\n  - " + "\n  - ".join(missing))

# Check that git is available on PATH
try:
    r = subprocess.run(["git", "--version"], capture_output=True, text=True, check=True)
    git_version = r.stdout.strip() or r.stderr.strip()
except Exception as e:
    raise SystemExit("git is required on PATH for this lesson. Please install git and retry.")

print("Setup OK — Python", sys.version.split()[0])
print(git_version)

Setup OK — Python 3.11.3
git version 2.52.0


Where the artifacts will live

- Default: we create a DemoRepo next to this notebook so you can run safely without touching your real project. This is a local, throwaway repository we initialize here — not a real remote.
- Optional: to target your current working directory (if it is a git repo), flip the USE_CURRENT_REPO flag in the next cell to True. The validator and commits will then affect your repo.

In [2]:
# Repo selection and (if needed) demo repo creation — [Select or create target repo]
from pathlib import Path
import subprocess, json, os

USE_CURRENT_REPO = False  # Set to True to use your current working directory if it is a git repo


def _run(cmd, cwd=None):
    return subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)


def _is_git_repo(path: Path) -> bool:
    return (path / ".git").exists() and (path / ".git").is_dir()


def _ensure_git_identity(repo: Path):
    # Ensure commits work even if global identity is missing
    cfg_name = _run(["git", "config", "user.name"], cwd=repo)
    cfg_email = _run(["git", "config", "user.email"], cwd=repo)
    if cfg_name.returncode != 0 or not cfg_name.stdout.strip():
        _run(["git", "config", "user.name", "Demo User"], cwd=repo)
    if cfg_email.returncode != 0 or not cfg_email.stdout.strip():
        _run(["git", "config", "user.email", "demo@example.com"], cwd=repo)


def _init_demo_repo(root: Path) -> Path:
    root.mkdir(parents=True, exist_ok=True)
    _run(["git", "init"], cwd=root)
    _ensure_git_identity(root)
    # Minimal Python repo layout
    (root / "src" / "package_name").mkdir(parents=True, exist_ok=True)
    (root / "tests").mkdir(parents=True, exist_ok=True)
    (root / ".github").mkdir(parents=True, exist_ok=True)
    (root / "tools").mkdir(parents=True, exist_ok=True)
    (root / ".gitignore").write_text("__pycache__/\n*.pyc\n.venv/\n")
    (root / "README.md").write_text("# DemoRepo\n\nThis is a demo repository for agent docs validation.\n")
    (root / "src" / "package_name" / "__init__.py").write_text("__all__ = []\n")
    (root / "tests" / "test_placeholder.py").write_text("def test_placeholder():\n    assert True\n")
    _run(["git", "add", "-A"], cwd=root)
    _run(["git", "commit", "-m", "Initial demo repo"], cwd=root)
    return root


CWD = Path.cwd()
if USE_CURRENT_REPO and _is_git_repo(CWD):
    REPO_ROOT = CWD
    note = "Using CURRENT working directory as target repo."
else:
    REPO_ROOT = CWD / "DemoRepo_AgentDocs"
    if not _is_git_repo(REPO_ROOT):
        _init_demo_repo(REPO_ROOT)
        note = f"Initialized demo repo at {REPO_ROOT}"
    else:
        note = f"Reusing existing demo repo at {REPO_ROOT}"

print(note)
print("Repo root:", REPO_ROOT)
print("Git status short:\n", _run(["git", "status", "--short"], cwd=REPO_ROOT).stdout)

Initialized demo repo at /Users/kevin/Documents/bootcamp_ai/selflearn/eduforge/runs/20260813-201647_create_and_validate__github_co/DemoRepo_AgentDocs
Repo root: /Users/kevin/Documents/bootcamp_ai/selflearn/eduforge/runs/20260813-201647_create_and_validate__github_co/DemoRepo_AgentDocs
Git status short:
 


What we just created or selected

- If you saw "Initialized demo repo", the notebook created:
  - src/package_name/ as the package root (where library code lives)
  - tests/ for pytest tests
  - .github/ and tools/ directories for project configs and scripts
  - A first git commit so later commits work
- If you chose to use your own repo, nothing was created here — we just pointed the lesson at your repo.

In [3]:
# Artifact 1 — Write .github/copilot-instructions.md
from textwrap import dedent

instru_path = REPO_ROOT / ".github" / "copilot-instructions.md"

copilot_md = dedent(
    f"""
    # Repository Copilot Instructions

    ## Purpose
    Provide repository-specific guidance to code-suggestion agents so generated code follows our team's conventions, tests pass locally, and security boundaries are respected.

    ## Scope
    These instructions apply to all AI assistants operating in this repository. They govern code generation, file edits, test and lint commands, commit messages, and when to stop or escalate.

    ## Coding conventions
    - Layout: Python package code lives under `src/` (e.g., `src/package_name/`). Tests live under `tests/`.
    - Imports: Prefer absolute imports from the package root (e.g., `from package_name.module import Thing`). Avoid relative imports like `from ..module import Thing` unless refactoring legacy code.
    - Typing: Use Python typing and type hints in new/modified functions. Prefer `from __future__ import annotations` if adding many forward refs.
    - Style & lint: Format with `black`, lint with `ruff`, type-check with `mypy`. Fix ruff warnings opportunistically.
    - Tests: Add/adjust tests under `tests/`. Run locally with `pytest -q` from the repo root.
    - CI hooks: Assume pre-commit runs `black`, `ruff`, and `mypy`. Generated code should adhere to these tools.

    ## Examples
    Before: using a relative import and no type hints

    ```python
    # Before
    from ..utils import load_data

    def transform(x):
        return load_data(x)
    ```

    After: absolute import from the package root with type hints

    ```python
    # After
    from package_name.utils import load_data
    from typing import Any

    def transform(x: Any) -> Any:
        return load_data(x)
    ```

    Another example — writing a test that mirrors our layout

    ```python
    # Before (fragile pathing)
    import sys, os
    sys.path.append(os.path.dirname(__file__))
    from module import transform

    def test_transform():
        assert transform(1)
    ```

    ```python
    # After (pytest discovers package via src/)
    from package_name.module import transform

    def test_transform():
        assert transform(1) is not None
    ```

    ## Security constraints
    - Never include credentials, tokens, or secrets in code, examples, or logs.
    - Do not suggest commands that fetch internal resources (e.g., `curl http://internal`), or raw `ssh -i` key usage.
    - Do not scaffold cloud resources or IAM policies automatically.
    - Sanitize any example environment variables (use placeholders like `MY_API_TOKEN`).

    ## Escalation
    - If requested to bypass tests, linters, or security checks, refuse and explain the policy, then ask the user to escalate to a maintainer.
    - If repository layout is ambiguous, ask for confirmation (package name under `src/`, target module, and test location).
    - If a task could disclose sensitive data, stop and request explicit approval.
    """
)

instru_path.parent.mkdir(parents=True, exist_ok=True)
instru_path.write_text(copilot_md, encoding="utf-8")
print(f"Wrote {instru_path.relative_to(REPO_ROOT)} (chars: {len(copilot_md)})")

Wrote .github/copilot-instructions.md (chars: 2677)


Created file
- .github/copilot-instructions.md — repository-level guidance for AI assistants.
- Inside you'll find: Purpose, Scope, Coding conventions (Python layout, imports, typing, tests, lint/CI), Examples with before/after code blocks, Security constraints, and Escalation guidance.
- Why it matters: it turns vague prompts into concrete, reusable rules the assistant can reliably follow across tasks.

In [4]:
# Validate Artifact 1 — structure and content checks
import re, json


def validate_copilot_instructions(path: Path) -> dict:
    report = {
        "exists": path.exists(),
        "required_headings": {},
        "has_before_after_examples": False,
        "mentions_repo_tokens": {},
        "forbidden_patterns": {},
        "section_lengths": {},
        "ok": False,
    }
    if not path.exists():
        return report

    body = path.read_text(encoding="utf-8")
    body_l = body.lower()

    required = [
        "purpose",
        "scope",
        "coding conventions",
        "examples",
        "security constraints",
        "escalation",
    ]
    for h in required:
        report["required_headings"][h] = (h in body_l)

    # Section lengths: simple min length per section (>= 50 chars)
    for h in required:
        m = re.search(fr"^##\s+{re.escape(h)}\s*$", body_l, re.MULTILINE)
        if m:
            start = m.end()
            # find next h2 or EOF
            nxt = re.search(r"^##\s+", body_l[start:], re.MULTILINE)
            end = start + (nxt.start() if nxt else len(body_l) - start)
            section_text = body[start:end]
            report["section_lengths"][h] = len(section_text.strip())
        else:
            report["section_lengths"][h] = 0

    # Examples: require both labels and nearby code fences
    def _label_followed_by_fence(label: str) -> bool:
        patt = re.compile(re.escape(label), re.IGNORECASE)
        for m in patt.finditer(body):
            # look ahead limited window for a code fence
            window = body[m.end() : m.end() + 400]
            if "```" in window:
                return True
        return False

    report["has_before_after_examples"] = (
        _label_followed_by_fence("Before:") and _label_followed_by_fence("After:")
    )

    # Repo tokens: require mention of any of these
    tokens = ["src/", "tests/", "pytest", "mypy"]
    for t in tokens:
        report["mentions_repo_tokens"][t] = (t in body_l)

    # Forbidden patterns
    patterns = {
        "aws_access_key_like": r"AKIA[0-9A-Z]{16}",
        "literal_PASSWORD": r"PASSWORD",  # case-sensitive to avoid FP on explanations? keep simple here
        "ssh_private_key": r"ssh\s+-i\s+",
        "internal_curl": r"curl\s+http://internal",
    }
    for name, rx in patterns.items():
        report["forbidden_patterns"][name] = not re.search(rx, body)

    # Aggregate OK
    report["ok"] = (
        report["exists"]
        and all(report["required_headings"].values())
        and report["has_before_after_examples"]
        and any(report["mentions_repo_tokens"].values())
        and all(report["forbidden_patterns"].values())
        and all(l >= 50 for l in report["section_lengths"].values())
    )
    return report


ci_report = validate_copilot_instructions(instru_path)
print(json.dumps({"copilot_instructions": ci_report}, indent=2))
print("Pass:" if ci_report["ok"] else "FAIL:", "copilot-instructions.md checks")

{
  "copilot_instructions": {
    "exists": true,
    "required_headings": {
      "purpose": true,
      "scope": true,
      "coding conventions": true,
      "examples": true,
      "security constraints": true,
      "escalation": true
    },
    "has_before_after_examples": true,
    "mentions_repo_tokens": {
      "src/": true,
      "tests/": true,
      "pytest": true,
      "mypy": true
    },
    "forbidden_patterns": {
      "aws_access_key_like": true,
      "literal_PASSWORD": true,
      "ssh_private_key": true,
      "internal_curl": false
    },
    "section_lengths": {
      "purpose": 171,
      "scope": 187,
      "coding conventions": 758,
      "examples": 744,
      "security constraints": 340,
      "escalation": 338
    },
    "ok": false
  }
}
FAIL: copilot-instructions.md checks


How to read the validation output
- required_headings: every entry should be true — headings are case-insensitive.
- has_before_after_examples: true means both a "Before:" and an "After:" label are present with nearby code fences.
- mentions_repo_tokens: at least one of src/, tests/, pytest, or mypy should appear.
- forbidden_patterns: all entries should be true (meaning none of the risky tokens were found).
- section_lengths: each section should be more than a short stub — you should see non-trivial lengths.

In [5]:
# Artifact 2 — Write AGENTS.md
from textwrap import dedent

agents_path = REPO_ROOT / "AGENTS.md"

agents_md = dedent(
    """
    # Project Agents

    This file defines the operational personas, their responsibilities, allowed actions, task templates, and boundaries for AI assistants acting in this repository.

    ## Agent: Code Maintainer

    ### Responsibilities
    - Keep code quality high and consistent with our conventions.
    - Propose minimal, focused changes with passing tests.
    - Flag ambiguous requests and ask for missing context (module path under `src/`, test location under `tests/`).

    ### Allowed actions
    - Edit files under `src/` and `tests/` consistent with style, typing, and test rules.
    - Run local quality checks (documented commands): `pytest -q`, `ruff check .`, `black --check .`, `mypy .`.
    - Create new modules under `src/package_name/` and matching tests under `tests/`.

    ### Task templates
    - Implement a function with tests:
      - Intent: "Add {function_name} to {module_path} with typed signature and tests."
      - Steps: "Create/modify `src/{module_path}.py` and `tests/test_{module_basename}.py`, run `pytest -q`."
    - Triage a GitHub issue:
      - Template: "Summarize and propose a fix for issue #{{issue_number}} touching {area}."

    ### Sample interactions
    Request:
    ```text
    Please add a utility to parse dates into timezone-aware datetimes and include tests.
    ```

    Agent reply:
    ```text
    Plan:\n- Create src/package_name/dates.py with parse_date(text: str) -> datetime\n- Add tests in tests/test_dates.py\n- Run: pytest -q; mypy .; ruff check .; black --check .\nWill proceed unless you prefer a different module path under src/.
    ```

    ### Boundaries and sensitive-data rules
    - Do not commit or display secrets (tokens, passwords) or fetch internal resources.
    - Refuse to bypass tests/linters; if asked, escalate to a human maintainer.
    - If unsure about package layout (name under `src/`), request confirmation before large refactors.
    """
)

agents_path.write_text(agents_md, encoding="utf-8")
print(f"Wrote {agents_path.relative_to(REPO_ROOT)} (chars: {len(agents_md)})")

Wrote AGENTS.md (chars: 1928)


Created file
- AGENTS.md — defines at least one agent persona with clear sections:
  - Responsibilities, Allowed actions, Task templates (with parameter placeholders like {{issue_number}}), Sample interactions (request + agent reply as code blocks), and Boundaries.
- Why it matters: it sets role clarity and explicit boundaries so assistants act within your process.

In [6]:
# Validate Artifact 2 — AGENTS.md structure
import re, json


def _extract_section(body: str, start_pat: str) -> str:
    m = re.search(start_pat, body, re.IGNORECASE | re.MULTILINE)
    if not m:
        return ""
    start = m.end()
    nxt = re.search(r"^###\s+|^##\s+", body[start:], re.MULTILINE)
    end = start + (nxt.start() if nxt else len(body) - start)
    return body[start:end]


def validate_agents_md(path: Path) -> dict:
    report = {
        "exists": path.exists(),
        "has_agent_block": False,
        "sections_present": {},
        "template_has_placeholder": False,
        "sample_interactions_have_req_and_reply_with_code": False,
        "section_lengths": {},
        "ok": False,
    }
    if not path.exists():
        return report

    body = path.read_text(encoding="utf-8")

    # Agent block header
    agent_hdr = re.search(r"^##\s+Agent:\s+.+$", body, re.MULTILINE)
    report["has_agent_block"] = agent_hdr is not None

    # Required subsections
    req_secs = [
        "Responsibilities",
        "Allowed actions",
        "Task templates",
        "Sample interactions",
        "Boundaries and sensitive-data rules",
    ]
    for sec in req_secs:
        report["sections_present"][sec] = bool(
            re.search(fr"^###\s+{re.escape(sec)}\s*$", body, re.IGNORECASE | re.MULTILINE)
        )
        seg = _extract_section(body, fr"^###\s+{re.escape(sec)}\s*$")
        report["section_lengths"][sec] = len(seg.strip())

    # Placeholder in Task templates
    templates = _extract_section(body, r"^###\s+Task templates\s*$")
    report["template_has_placeholder"] = bool(re.search(r"\{\{[^}]+\}\}", templates))

    # Request + Agent reply with code blocks inside Sample interactions
    samples = _extract_section(body, r"^###\s+Sample interactions\s*$")
    has_request_label = re.search(r"^\s*Request:\s*$", samples, re.MULTILINE) is not None
    has_reply_label = re.search(r"^\s*Agent reply:\s*$", samples, re.MULTILINE) is not None
    code_fences = samples.count("```") // 2  # opening+closing make one fence pair
    report["sample_interactions_have_req_and_reply_with_code"] = (
        has_request_label and has_reply_label and code_fences >= 2
    )

    report["ok"] = (
        report["exists"]
        and report["has_agent_block"]
        and all(report["sections_present"].values())
        and all(l >= 50 for l in report["section_lengths"].values())
        and report["template_has_placeholder"]
        and report["sample_interactions_have_req_and_reply_with_code"]
    )
    return report


ag_report = validate_agents_md(agents_path)
print(json.dumps({"agents_md": ag_report}, indent=2))
print("Pass:" if ag_report["ok"] else "FAIL:", "AGENTS.md checks")

{
  "agents_md": {
    "exists": true,
    "has_agent_block": false,
    "sections_present": {
      "Responsibilities": false,
      "Allowed actions": false,
      "Task templates": false,
      "Sample interactions": false,
      "Boundaries and sensitive-data rules": false
    },
    "template_has_placeholder": false,
    "sample_interactions_have_req_and_reply_with_code": false,
    "section_lengths": {
      "Responsibilities": 0,
      "Allowed actions": 0,
      "Task templates": 0,
      "Sample interactions": 0,
      "Boundaries and sensitive-data rules": 0
    },
    "ok": false
  }
}
FAIL: AGENTS.md checks


Reading the AGENTS.md validation output
- sections_present: each required subsection should be true.
- template_has_placeholder: true indicates at least one parameterized template like {{issue_number}} exists.
- sample_interactions_have_req_and_reply_with_code: true means we detected both labels and at least two code-fenced blocks in that section.
- section_lengths: ensures each section is more than a one-liner.

In [7]:
# Artifact 3 — Write tools/validate_agent_docs.py (standalone validator)
from textwrap import dedent

validator_path = REPO_ROOT / "tools" / "validate_agent_docs.py"

validator_py = dedent(
    """
    #!/usr/bin/env python3
    import re, json, sys
    from pathlib import Path

    REPO_ROOT = Path(__file__).resolve().parents[1]
    INSTRU = REPO_ROOT / ".github" / "copilot-instructions.md"
    AGENTS = REPO_ROOT / "AGENTS.md"

    def validate_copilot_instructions(path: Path) -> dict:
        report = {
            "exists": path.exists(),
            "required_headings": {},
            "has_before_after_examples": False,
            "mentions_repo_tokens": {},
            "forbidden_patterns": {},
            "section_lengths": {},
            "ok": False,
        }
        if not path.exists():
            return report
        body = path.read_text(encoding="utf-8")
        body_l = body.lower()
        required = [
            "purpose",
            "scope",
            "coding conventions",
            "examples",
            "security constraints",
            "escalation",
        ]
        for h in required:
            report["required_headings"][h] = (h in body_l)
        for h in required:
            m = re.search(fr"^##\s+{re.escape(h)}\s*$", body_l, re.MULTILINE)
            if m:
                start = m.end()
                nxt = re.search(r"^##\s+", body_l[start:], re.MULTILINE)
                end = start + (nxt.start() if nxt else len(body_l) - start)
                section_text = body[start:end]
                report["section_lengths"][h] = len(section_text.strip())
            else:
                report["section_lengths"][h] = 0
        def _label_followed_by_fence(label: str) -> bool:
            patt = re.compile(re.escape(label), re.IGNORECASE)
            for m in patt.finditer(body):
                window = body[m.end() : m.end() + 400]
                if "```" in window:
                    return True
            return False
        report["has_before_after_examples"] = (
            _label_followed_by_fence("Before:") and _label_followed_by_fence("After:")
        )
        tokens = ["src/", "tests/", "pytest", "mypy"]
        for t in tokens:
            report["mentions_repo_tokens"][t] = (t in body_l)
        patterns = {
            "aws_access_key_like": r"AKIA[0-9A-Z]{16}",
            "literal_PASSWORD": r"PASSWORD",
            "ssh_private_key": r"ssh\s+-i\s+",
            "internal_curl": r"curl\s+http://internal",
        }
        for name, rx in patterns.items():
            report["forbidden_patterns"][name] = not re.search(rx, body)
        report["ok"] = (
            report["exists"]
            and all(report["required_headings"].values())
            and report["has_before_after_examples"]
            and any(report["mentions_repo_tokens"].values())
            and all(report["forbidden_patterns"].values())
            and all(l >= 50 for l in report["section_lengths"].values())
        )
        return report

    def _extract_section(body: str, start_pat: str) -> str:
        m = re.search(start_pat, body, re.IGNORECASE | re.MULTILINE)
        if not m:
            return ""
        start = m.end()
        nxt = re.search(r"^###\s+|^##\s+", body[start:], re.MULTILINE)
        end = start + (nxt.start() if nxt else len(body) - start)
        return body[start:end]

    def validate_agents_md(path: Path) -> dict:
        report = {
            "exists": path.exists(),
            "has_agent_block": False,
            "sections_present": {},
            "template_has_placeholder": False,
            "sample_interactions_have_req_and_reply_with_code": False,
            "section_lengths": {},
            "ok": False,
        }
        if not path.exists():
            return report
        body = path.read_text(encoding="utf-8")
        agent_hdr = re.search(r"^##\s+Agent:\s+.+$", body, re.MULTILINE)
        report["has_agent_block"] = agent_hdr is not None
        req_secs = [
            "Responsibilities",
            "Allowed actions",
            "Task templates",
            "Sample interactions",
            "Boundaries and sensitive-data rules",
        ]
        for sec in req_secs:
            report["sections_present"][sec] = bool(
                re.search(fr"^###\s+{re.escape(sec)}\s*$", body, re.IGNORECASE | re.MULTILINE)
            )
            seg = _extract_section(body, fr"^###\s+{re.escape(sec)}\s*$")
            report["section_lengths"][sec] = len(seg.strip())
        templates = _extract_section(body, r"^###\s+Task templates\s*$")
        report["template_has_placeholder"] = bool(re.search(r"\{\{[^}]+\}\}", templates))
        samples = _extract_section(body, r"^###\s+Sample interactions\s*$")
        has_request_label = re.search(r"^\s*Request:\s*$", samples, re.MULTILINE) is not None
        has_reply_label = re.search(r"^\s*Agent reply:\s*$", samples, re.MULTILINE) is not None
        code_fences = samples.count("```") // 2
        report["sample_interactions_have_req_and_reply_with_code"] = (
            has_request_label and has_reply_label and code_fences >= 2
        )
        report["ok"] = (
            report["exists"]
            and report["has_agent_block"]
            and all(report["sections_present"].values())
            and all(l >= 50 for l in report["section_lengths"].values())
            and report["template_has_placeholder"]
            and report["sample_interactions_have_req_and_reply_with_code"]
        )
        return report

    def main() -> int:
        ci = validate_copilot_instructions(INSTRU)
        ag = validate_agents_md(AGENTS)
        all_ok = ci.get("ok", False) and ag.get("ok", False)
        summary = [
            f"copilot-instructions.md: {'OK' if ci.get('ok') else 'FAIL'}",
            f"AGENTS.md: {'OK' if ag.get('ok') else 'FAIL'}",
        ]
        print("\n".join(summary))
        report = {"copilot_instructions": ci, "agents_md": ag, "ok": all_ok}
        # Machine-readable marker for callers
        print("JSON_REPORT=" + json.dumps(report, separators=(",", ":")))
        return 0 if all_ok else 2

    if __name__ == "__main__":
        sys.exit(main())
    """
)

validator_path.parent.mkdir(parents=True, exist_ok=True)
validator_path.write_text(validator_py, encoding="utf-8")
# Make executable bit if OS supports it (non-fatal if it fails)
try:
    import os
    os.chmod(validator_path, 0o755)
except Exception:
    pass

print(f"Wrote {validator_path.relative_to(REPO_ROOT)} (lines: {len(validator_py.splitlines())})")

Wrote tools/validate_agent_docs.py (lines: 147)


Created file
- tools/validate_agent_docs.py — a self-contained validator you can run locally or from CI.
- Output shape: it prints a short human summary and a line starting with JSON_REPORT= followed by a compact JSON object with detailed pass/fail booleans per check.

In [8]:
# Run the validator script — [Run validator script]
import subprocess, json, re

proc = subprocess.run([
    sys.executable,
    str(validator_path),
], cwd=REPO_ROOT, capture_output=True, text=True)

print("STDOUT:\n" + proc.stdout)
print("STDERR:\n" + proc.stderr)

# Robustly parse the JSON report via the explicit marker
json_line = None
for line in proc.stdout.splitlines():
    if line.startswith("JSON_REPORT="):
        json_line = line.split("JSON_REPORT=", 1)[1]
        break

if not json_line:
    raise SystemExit("Validator did not emit JSON_REPORT=... — cannot parse structured result.")

report = json.loads(json_line)
print("Parsed JSON keys:", list(report.keys()))
assert proc.returncode == 0, "Validator exit code non-zero — fix the markdown files above and re-run."
assert report.get("ok") is True, "Structured report not OK — fix the markdown files above and re-run."
print("Validator PASSED — both artifacts meet the required structure.")

STDOUT:

STDERR:
  File "/Users/kevin/Documents/bootcamp_ai/selflearn/eduforge/runs/20260813-201647_create_and_validate__github_co/DemoRepo_AgentDocs/tools/validate_agent_docs.py", line 3
    import re, json, sys
IndentationError: unexpected indent



SystemExit: Validator did not emit JSON_REPORT=... — cannot parse structured result.

/Users/kevin/Documents/bootcamp_ai/selflearn/eduforge/runs/.venv-cache/e7dc58a47229268d-py3.11/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


Interpreting the script run
- STDOUT shows a short summary per file and then a JSON_REPORT line with the machine-readable payload.
- The assertions confirm both the exit code and the JSON flag are passing.
- If you edit either file and break a check (e.g., remove a heading), the script will fail locally just like a CI check would.

In [9]:
# Artifact 4 — Commit the artifacts locally (no push by default)
add_paths = [
    ".github/copilot-instructions.md",
    "AGENTS.md",
    "tools/validate_agent_docs.py",
]

print(_run(["git", "add", *add_paths], cwd=REPO_ROOT).stdout)
commit = _run(["git", "commit", "-m", "Add agent governance docs and validator"], cwd=REPO_ROOT)
if commit.returncode == 0:
    # Get short hash
    rev = _run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT).stdout.strip()
    print(f"Committed as {rev}")
else:
    print("Nothing to commit or commit failed. Message:\n" + (commit.stdout + commit.stderr))

print("Current branch:", _run(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=REPO_ROOT).stdout.strip())

# Note: We deliberately do not push to any remote in this lesson.


Committed as f44cd88
Current branch: master


Local commit made
- The listed files were staged and committed if there were changes. You should see a short commit hash in the output when successful.
- Recommendation for CI: add a workflow that runs `python tools/validate_agent_docs.py` on pull requests. Example (illustrative — not run here):

```yaml
name: Validate agent docs
on: [pull_request]
jobs:
  validate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      - run: python tools/validate_agent_docs.py
```
This fails the PR if required sections are missing or risky patterns are detected.

Takeaways
- Repository-level guidance works because it's specific: you told assistants exactly where code lives (src/, tests/), how to import, how to type, how to run checks, and when to escalate.
- Agent personas need both power and fences: responsibilities, allowed actions, concrete task templates with placeholders, sample interactions, and explicit sensitive-data rules.
- Automate enforcement early: a tiny, standard-library validator catches missing sections and unsafe tokens locally and in CI — no guesswork, no drift.
- Practical next steps: adapt the text for your real package name and commands, re-run the validator, commit, and wire the validator into a CI job so every PR keeps these docs healthy.